# Rekord Box Tag Inference


In [6]:
# automatically reload imported modules before executing code

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from nbutils import setup_path

setup_path()

In [8]:
from pyrekordbox import Rekordbox6Database
from pyrekordbox.db6 import tables

# Get table refs 
song_table = tables.DjmdContent
tag_table = tables.DjmdMyTag
song_tag_table = tables.DjmdSongMyTag

In [4]:
from utils import get_db_content

db = Rekordbox6Database()

raw_df = get_db_content(db)

In [9]:
raw_df.head(10)

ContentID,FolderPath,Title,ArtistID,ArtistName,GenreID,GenreName,BPM,DateCreated,Length,MyTagNames,MyTagIDs
str,str,str,str,str,str,str,i32,str,i32,list[str],list[str]
"""36085625""","""/Users/quintenrosseel/Music/Pi…","""Surrender""","""1673664596""","""Gerd Janson""","""159652946""","""Techno/House""",12200,"""2021-12-26""",413,[],[]
"""18988943""","""/Users/quintenrosseel/Music/Pi…","""I Want Your Soul""","""1642798025""","""Armand Van Helden""","""3027895697""","""Classics/House""",12800,"""2021-12-26""",399,[],[]
"""118919255""","""/Users/quintenrosseel/Music/Pi…","""Save Our Love""","""2508199194""","""Escape From New York""","""3584637391""","""Disco""",11010,"""2022-08-23""",304,"[""80s"", ""Disco"", … ""Chill""]","[""1155275430"", ""62609778"", … ""3718426834""]"
"""228400738""","""/Users/quintenrosseel/Music/Pi…","""RENT4""","""3953188055""","""Lakim""","""4282047218""","""Garage""",14500,"""2021-07-17""",167,[],[]
"""217615930""","""/Users/quintenrosseel/Music/Pi…","""Funky Child (1993)""","""1137483595""","""Lords Of The Underground""","""2864790501""","""Hip-hop""",9630,"""2018-10-11""",227,"[""Beats"", ""Hip-Hop"", … ""TAG YEAR""]","[""2053683127"", ""3917148722"", … ""382831235""]"
"""121941058""","""/Users/quintenrosseel/Music/Pi…","""I Wanna Do The Do (Extended Re…","""3249033290""","""Bobby Rush""","""1520704553""","""Funk/Soul/Motown/Jazz""",10990,"""2018-10-11""",390,[],[]
"""261795999""","""/Users/quintenrosseel/Music/Pi…","""Days Like This (DJ Spinna Remi…","""645442704""","""Shaun Escoffery""","""169675979""","""Disco/House""",11990,"""2018-10-11""",486,[],[]
"""191218482""","""/Users/quintenrosseel/Music/Pi…","""Dancer""","""276731648""","""Gino Soccio""","""3584637391""","""Disco""",12190,"""2019-01-11""",507,[],[]
"""55055504""","""/Users/quintenrosseel/Music/Pi…","""Untitled 02""","""4081332306""","""Unknown Artist""","""1935845763""","""Warmup/House""",12300,"""2018-10-11""",475,"[""Burning Man"", ""Loungy"", … ""Trippy""]","[""169548333"", ""4144163695"", … ""938485439""]"


# Simple Tagging Algorithm: Artist Propagation

A simple technique for tagging:

1. look at tags of other songs from a given artist
2. If a tag exist on any song of that artist, apply that same tag to the other untagged songs. 

177 such tags can be applied. 

In [15]:
import polars as pl

processed_df = (
    raw_df
    .with_columns((pl.col("MyTagNames").list.len() > 0) .alias("has_tags"))
)

processed_df.head()

ContentID,FolderPath,Title,ArtistID,ArtistName,GenreID,GenreName,BPM,DateCreated,Length,MyTagNames,MyTagIDs,has_tags
str,str,str,str,str,str,str,i32,str,i32,list[str],list[str],bool
"""36085625""","""/Users/quintenrosseel/Music/Pi…","""Surrender""","""1673664596""","""Gerd Janson""","""159652946""","""Techno/House""",12200,"""2021-12-26""",413,[],[],false
"""18988943""","""/Users/quintenrosseel/Music/Pi…","""I Want Your Soul""","""1642798025""","""Armand Van Helden""","""3027895697""","""Classics/House""",12800,"""2021-12-26""",399,[],[],false
"""118919255""","""/Users/quintenrosseel/Music/Pi…","""Save Our Love""","""2508199194""","""Escape From New York""","""3584637391""","""Disco""",11010,"""2022-08-23""",304,"[""80s"", ""Disco"", … ""Chill""]","[""1155275430"", ""62609778"", … ""3718426834""]",true
"""228400738""","""/Users/quintenrosseel/Music/Pi…","""RENT4""","""3953188055""","""Lakim""","""4282047218""","""Garage""",14500,"""2021-07-17""",167,[],[],false
"""217615930""","""/Users/quintenrosseel/Music/Pi…","""Funky Child (1993)""","""1137483595""","""Lords Of The Underground""","""2864790501""","""Hip-hop""",9630,"""2018-10-11""",227,"[""Beats"", ""Hip-Hop"", … ""TAG YEAR""]","[""2053683127"", ""3917148722"", … ""382831235""]",true


In [24]:
artist_tag_stats = (
    processed_df
    .group_by("ArtistID", "ArtistName", "has_tags")
    .agg(pl.len().alias("count"))
    .pivot(values="count", index=["ArtistID", "ArtistName"], on="has_tags")
    .filter(
        pl.col("true").is_not_null() & pl.col("false").is_not_null()
    )
    .with_columns(
        total=pl.col("true") + pl.col("false")
    )
    .sort("total", descending=True)
)

artist_tag_stats

ArtistID,ArtistName,false,true,total
str,str,u32,u32,u32
"""2476807121""","""BICEP""",12,3,15
"""1908354044""","""Daft Punk""",8,2,10
null,null,8,1,9
"""3307290350""","""LAKIM""",8,1,9
"""2388580169""","""Mansur Brown""",5,4,9
…,…,…,…,…
"""2361723307""","""Les Yeux Orange""",1,1,2
"""1803051969""","""Lio""",1,1,2
"""4205567497""","""Ayla""",1,1,2


In [26]:
artist_tag_stats['false'].sum()

177